# Прогнозирование и калибровка CTR для рекламной платформы Advandex

Автор: Якунин Михаил

Дата: 24 мая 2026 г.

***По этой ссылке можно посмотреть исходный код проекта и скачать последнюю версию: https://github.com/MEYakunin/Sprint12_CTR_prediction.git***

## Цель проекта

Построить бинарный классификатор (SVM и логистическая регрессия) для предсказания вероятности клика по рекламному объявлению и выполнить калибровку предсказаний, чтобы обеспечить соответствие между предсказанными вероятностями и реальной частотой кликов.

## Задача проекта

* Обработать и проанализировать анонимизированные данные о показах рекламы.
* Отобрать наиболее информативные признаки с помощью фильтрационных методов и методов-обёрток.
* Обучить SVM с линейным ядром и логистическую регрессию.
* Настроить гиперпараметры через GridSearchCV.
* Провести калибровку вероятностей (изотоническая регрессия).
* Оценить модели по PR-AUC, Log Loss, Brier Score, ECE/MCE.

## Содержание <a id=content></a>

1. [Подготовка среды и загрузка данных](#1)
2. [Исследовательский анализ данных (EDA)](#2)
3. [Разделение данных на выборки](#3)
4. [Предобработка данных — построение пайплайнов](#4)
5. [Отбор признаков](#5)
6. [Обучение базовой модели](#6)
7. [Подбор гиперпараметров: Grid Search с кросс-валидацией](#7)
8. [Финальная модель](#8)
9. [Калибровка модели](#9)
10. [Оценка качества калибровки](#10)
11. [Финальный отчёт и выводы](#11)
12. [Сохранение модели для продакшена](#12)

## [Этап 1. Подготовка среды и загрузка данных <a id=1></a>](#content)

**В этом этапе мы:**
- импортируем библиотеки и настроим параметры отображения графиков и датафреймов;
- зафиксируем значение `RANDOM_SEED` для воспроизводимости результатов;
- загрузим исходные данные из CSV-файла;
- выполним первичный просмотр данных: размер, первые строки, типы столбцов, общая статистика.

In [ ]:
# Устанавливаем все необходимые библиотеки
#!pip install requirements.txt

In [ ]:
# Загружаем библиотеки
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import precision_score, recall_score, average_precision_score, make_scorer, f1_score, brier_score_loss
from sklearn.svm import SVC

from category_encoders import TargetEncoder


In [ ]:
# Настраиваем параметры отображения графиков и датафреймов
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

In [ ]:
# Фиксируем константные значения
random_state = 11

In [ ]:
# Загружаем датасет
df = pd.read_csv('datasets/ds_s16_ad_click_dataset.csv', sep=',', decimal='.')

In [ ]:
# Выводим общую информацию о датафрейме
display(df.head(5))
df.info()
df.describe(include='all').T

### Промежуточный итог по этапу 1:

1. Библиотеки успешно импортированы, заданы настройки отображения (`display.max_columns`, `plt.style.use`).
2. Зафиксирован `random_state = 11` — это обеспечит воспроизводимость при случайных разделениях и инициализациях моделей.
3. Данные загружены из файла `'/datasets/ds_s16_ad_click_dataset.csv'`.
4. Размер датасета: **50 000 строк, 34 столбца**.
5. Типы данных:
   - **числовые** (`int64`, `float64`) – 23 столбца;
   - **категориальные** (`object` / `str`) – 11 столбцов.
6. Пропусков в данных нет (Non-Null Count для всех столбцов равен 50 000).
7. Целевая переменная `click` имеет два значения (0/1), её распределение будет подробно исследовано на этапе EDA.

Все подготовительные действия выполнены, данные готовы к дальнейшему анализу.

## [Этап 2. Исследовательский анализ данных (EDA) <a id=2></a>](#content)

### 2.1 Анализ целевой переменной

In [ ]:
# Анализ распределения целевой переменной click
print('Распределение целевой переменной "click":')
display(df['click'].value_counts() \
                .reset_index() \
                .rename(columns={'click': 'Признак'}))

plt.figure(figsize=(6,4))
sns.countplot(data=df, x='click', hue='click', palette='Set2', legend=False)
plt.xlabel('Click (0 = нет клика, 1 = клик)')
plt.ylabel('Количество показов')
plt.show()

In [ ]:
# Дополнительно: посчитаем долю кликов в процентах
click_rate = df['click'].mean() * 100
print(f"\nДоля кликов (CTR) в датасете: {click_rate:.2f}%")

**Сильный дисбаланс классов:**  
  - Класс `0` (нет клика) значительно преобладает над классом `1` (клик).  
  - Доля кликов составляет примерно около `17%`. Это типичная ситуация для задач прогнозирования `CTR`.
  - Целевая переменная сильно несбалансирована, что обосновывает выбор `PR-AUC` в качестве основной метрики и требует стратифицированного разделения выборок.

### 2.2 Анализ пропущенных значений

In [ ]:
# Проверка кол-во пропусков в каждом признаке
df.isnull().sum()

Как видно из кода, **ни один из признаков не содержит пропущенных значений**. Все 33 признака (после удаления `id`, `hour`, `device_ip`) имеют полные данные. Отсутствие пропусков упрощает предобработку: не требуется применять `SimpleImputer` или удалять строки. Однако для универсальности кода и на случай появления пропусков в будущих данных (например, в продакшене) можно оставить imputer в пайплайне с настройками по умолчанию.

Если бы пропуски были, мы бы использовали:
- **Для числовых признаков:** медиана (устойчива к выбросам).
- **Для категориальных признаков:** константа `'unknown'` (сохраняет информацию о факте пропуска).

Пропуски отсутствуют, поэтому шаг заполнения пропусков можно пропустить или оставить imputer в пайплайне как запасной вариант (он не изменит данные, если пропусков нет).

### 2.3 Анализ признаков

In [ ]:
# Список всех признаков
print("Все признаки в датасете:")
print(df.columns.tolist())

In [ ]:
# Разделение по типам данных (фактические типы)
print("\nТипы данных:")
print(df.dtypes)

In [ ]:
# Определяем типы признаков
numeric_by_dtype = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_by_dtype = df.select_dtypes(include=['object', 'string']).columns.tolist()

print(f"\nЧисловые признаки (по dtype ['int64', 'float64']): {numeric_by_dtype}")
print(f"Категориальные признаки (по dtype ['object', 'string']): {categorical_by_dtype}")

In [ ]:
# Важное замечание: некоторые числовые признаки на самом деле категориальные (мало уникальных значений)
# Их мы будем обрабатывать как категориальные при кодировании
potential_categorical_numeric = ['C1', 'banner_pos', 'device_type', 'device_conn_type', 'C14', 'C15', 'C16', 'C17', 'C18', 'C19', 'C20', 'C21', 'ml_feature_4']
print(f"\nЧисловые признаки с малым числом уникальных значений (категориальные по сути):\n{potential_categorical_numeric}")

**Типы признаков:**
- **Категориальные (object):** `site_id`, `site_domain`, `site_category`, `app_id`, `app_domain`, `app_category`, `device_id`, `device_model`, `ml_feature_2`, `ml_feature_7`
- **Числовые (int64):** `id`, `click`, `C1`, `banner_pos`, `device_type`, `device_conn_type`, `C14`, `C15`, `C16`, `C17`, `C18`, `C19`, `C20`, `C21`, `ml_feature_1`, `ml_feature_3`, `ml_feature_4`, `ml_feature_5`, `ml_feature_6`, `ml_feature_8`, `ml_feature_9`, `ml_feature_10`

**Важные нюансы:**
- Некоторые числовые признаки имеют **очень мало уникальных значений** (например, `C1` – 7, `banner_pos` – 7, `device_type` – 4, `device_conn_type` – 4, `C14` - 1497, `C15` – 8, `C16` – 9, `C17` - 387, `C18` – 4, `C19` - 64, `C20` - 149, `C21` - 59, `ml_feature_4` - 2). По сути, они являются категориальными, но представлены числами. Их следует кодировать как категории (One-Hot Encoding), а не масштабировать.
- Признаки `ml_feature_1`, `ml_feature_3`, `ml_feature_5`, `ml_feature_6`, `ml_feature_8`, `ml_feature_9`, `ml_feature_10` имеют 50 000 уникальных значений (почти каждая строка уникальна). Это похоже на непрерывные или высокоразмерные числовые признаки. Их нужно масштабировать.
- `click` — целевая переменная (2 значения).

**Бесполезные признаки (удаляем сразу)**
1. **`id`** – уникальный идентификатор каждой записи. Не несёт предсказательной силы.
2. **`hour`** – временная метка. В чистом виде может привести к переобучению (модель запомнит конкретные часы). Можно было бы извлечь час дня, но в рамках первой итерации удалим.
3. **`device_ip`** – слишком много уникальных значений, не обобщается. Удаляем.

In [ ]:
# Удаляем бесполезные признаки
df_clean = df.drop(columns=['id', 'device_ip'])

### 2.4 Анализ категориальных признаков

In [ ]:
# Категориальные признаки: object
categorical_cols = df_clean.select_dtypes(include=['object', 'string']).columns.tolist()

# Добавляем числовые, которые являются категориальными
all_categorical = categorical_cols + potential_categorical_numeric

In [ ]:
# Создадим списки для признаков с малой и высокой кардинальностью 
low_categorical = []
high_categorical = []

# Анализ количества уникальных значений для каждого категориального признака
print("\nКоличество уникальных значений в категориальных признаках:")
for col in all_categorical:
    n_unique = df_clean[col].nunique()
    print(f"  {col}: {n_unique} уникальных значений")
    if n_unique > 50:
        high_categorical.append(col)
    else:
        low_categorical.append(col)

**Стратегии кодирования:**
- **One-Hot Encoding** — для признаков с числом уникальных значений ≤ 20 (или небольшое). Это: `site_category`, `app_domain`, `app_category`, `ml_feature_2`, `ml_feature_7`, а также все числовые категориальные (`C1`, `banner_pos`, `device_type`, `device_conn_type`, `C15`, `C16`, `C18`, `ml_feature_4`).  
  *Обоснование:* они имеют не более 22 категорий, One-Hot не приведёт к взрывному росту признаков.
- **Target Encoding** — для признаков с высокой кардинальностью (более 100 уникальных значений): `site_id`, `site_domain`, `app_id`, `device_id`, `device_model`, `C14`, `C17`, `C19`, `C20`, `C21`.  
  *Обоснование:* One-Hot создал бы тысячи новых столбцов, что неэффективно. Target Encoding заменяет каждую категорию на средний CTR по ней, что компактно и информативно.
 
Определены все категориальные признаки, включая скрытые в числовых. Выбраны методы кодирования.

### 2.5 Анализ числовых признаков (распределения, выбросы)

In [ ]:
# Определяем числовые признаки для масштабирования
numeric_cols = list(set(df_clean.columns) - set(all_categorical) - set(['click']))

In [ ]:
# Описательные статистики числовых признаков
print("\nСтатистики для числовых признаков:")
display(df_clean[numeric_cols].describe())

In [ ]:
# Визуализация распределений (гистограммы)
import matplotlib.pyplot as plt
fig, axes = plt.subplots(nrows=len(numeric_cols)//3 + 1, ncols=3, figsize=(15, 20))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    df[col].hist(bins=50, ax=axes[i], alpha=0.7, edgecolor='black')
    axes[i].set_title(col)
    axes[i].set_xlabel('Значение')
    axes[i].set_ylabel('Частота')
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Выбросы: используем IQR для каждого признака
print("\nАнализ выбросов (по IQR):")
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]
    print(f"{col}: {len(outliers)} выбросов ({len(outliers)/len(df)*100:.2f}%)")

**Ключевые наблюдения:**
- `ml_feature_3` имеет самый высокий стандартный разброс (std ≈ 5.79) и диапазон от -10 до 10.
- Остальные признаки имеют std близкий к 1 или меньше, диапазоны умеренные.
- Почти все признаки имеют 50 000 уникальных значений (кроме `ml_feature_4`), что говорит о высокой информативности и непрерывном характере.

**Вывод по выбросам:**
- В четырёх признаках (`ml_feature_1`, `ml_feature_5`, `ml_feature_9`, `ml_feature_10`) присутствуют выбросы на уровне ~0.7% от всех данных. Это небольшая доля.
- Признаки `ml_feature_3`, `ml_feature_6`, `ml_feature_8` не имеют выбросов по методу IQR.

**Стратегия обработки:**
1. **Масштабирование** — обязательно для всех 8 признаков. SVM чувствителен к масштабу признаков. Используем `StandardScaler`, который приводит данные к нулевому среднему и единичному стандартному отклонению. Это особенно важно для `ml_feature_3` с большим разбросом.
2. **Выбросы** — удалять не будем. Во-первых, их доля мала (менее 1%). Во-вторых, выбросы могут нести полезную информацию для модели. SVM с линейным ядром и регуляризацией (параметр C) достаточно устойчив к выбросам. Масштабирование уменьшит их влияние, но не устранит полностью. Если в дальнейшем качество модели будет низким, можно рассмотреть winsorization, но на старте оставляем как есть.

### 2.6 Анализ корреляции

In [ ]:
# Корреляция числовых признаков с целевой переменной 'click'
numeric_cols_corr = df_clean.select_dtypes(include=['int64', 'float64']).columns

print("Корреляция Пирсона числовых признаков с целевой переменной 'click':")
corr_with_target = df[numeric_cols_corr].corr()['click'].drop('click').sort_values(ascending=False)
print(corr_with_target)

In [ ]:
# Тепловая карта корреляций всех числовых признаков
plt.figure(figsize=(14, 12))
sns.heatmap(df_clean[numeric_cols_corr].corr(), 
            annot=True,     
            fmt='.2f',      
            cmap='coolwarm', 
            center=0,
            linewidths=0.5, 
            linecolor='white')
plt.title('Correlation Matrix of Numerical Features', fontsize=14)
plt.tight_layout()  
plt.show()

**Корреляция признаков с целевой переменной `click`**

Ни один из числовых признаков не имеет сильной линейной связи с целевой переменной. Это ожидаемо для CTR-задач, где сигнал слабый и требует нелинейных методов или комбинаций признаков. Тем не менее, признаки с наибольшей корреляцией (`ml_feature_9`, `ml_feature_10`, `C16`) могут оказаться полезными.

**Корреляции между признаками (мультиколлинеарность)**

Обнаружены **очень сильные линейные связи** между некоторыми признаками:

- `C1` и `device_type` → 0.899
- `C14` и `C17` → 0.976

**Мультиколлинеарность:**  
  При использовании линейного `SVM` с регуляризацией (параметр C) проблема не критична. Однако если мы будем применять методы-обёртки (`RFE`), они могут давать разные наборы признаков при наличии коррелированных групп. Будем осторожны.

  Корреляционный анализ выявил слабую линейную связь признаков с `click`, но сильную связь между самими признаками (`C1`–`device_type`, `C14`–`C17`). Это будет учтено при отборе признаков и настройке регуляризации моделей.

### 2.7 Промежуточный итог по EDA

**Ключевые находки**

- Данные не содержат пропусков, что упрощает предобработку.
- Сильный дисбаланс классов требует осторожного выбора метрик и стратификации.
- Признаки разделены на три группы:
  - непрерывные числовые (8 ml‑features) → масштабирование;
  - категориальные с малой кардинальностью → One‑Hot;
  - категориальные с высокой кардинальностью → Target Encoding.
- Выбросы в числовых признаках немногочисленны, их решено не удалять.
- Корреляция с целевой переменной низкая, что типично для CTR‑задач; основной вклад, вероятно, дадут высокоуровневые категориальные признаки после кодирования.

**Наиболее перспективные признаки для модели**

- По корреляции с `click`: `ml_feature_9`, `ml_feature_10`, `C16`.
- Высококардинальные категориальные признаки (`site_id`, `device_id` и др.) после Target Encoding могут быть очень информативны, так как они кодируются средним CTR по категории.
- Также стоит включить все признаки, удалив только явно бесполезные (`id`, `hour`, `device_ip`).

**Действия по предобработке**

- Удалить `id`, `hour`, `device_ip` (уже сделано).
- Масштабировать 7 числовых ml‑features через `StandardScaler`.
- Применить One‑Hot Encoding для категориальных признаков с малым числом уникальных значений (список выше).
- Применить Target Encoding для высококардинальных категорий (`site_id`, `site_domain`, `app_id`, `app_domain`, `device_id`, `device_model` и т.д.).
- Объединить все преобразования в единый `ColumnTransformer` внутри `Pipeline`, чтобы избежать утечки данных.

**Дополнительные замечания**

- Поскольку пропуски отсутствуют, `SimpleImputer` можно не включать, но для надёжности оставим его с параметрами по умолчанию (он не изменит данные).
- При использовании Target Encoding необходимо обучать его только на тренировочных данных (это автоматически обеспечит `Pipeline`).
- Результат: данные полностью подготовлены для построения пайплайнов, отбора признаков и последующего обучения моделей.

## [Этап 3. Разделение данных на выборки <a id=3></a>](#content)

#### 3.1 Разделите данные
- Сначала отделите тестовую выборку, в ней должно быть 20% данных.
- Оставшиеся 80% данных используйте для обучения.
- Используйте стратифицированное разделение, чтобы сохранить баланс классов.
- **Не используйте тестовую выборку до финального тестирования!**

#### 3.2 Проверьте разделение
- Убедитесь, что распределение целевой переменной сохранено в каждой выборке.
- Выведите размеры выборок.

In [ ]:
# Определяем признаки и целевую переменную
X = df_clean.drop(columns= ['click'])
y = df_clean['click']

In [ ]:
# Разделяем данные: 80% - обучение, 20% - тест
# Используем стратификацию для сохранения баланса классов
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=random_state,
    stratify=y
)

# Проверяем размеры выборок
print("Размер обучающей выборки:", X_train.shape)
print("Размер тестовой выборки:", X_test.shape)
print("\n")

In [ ]:
# Проверяем распределение целевой переменной
print("\nПроверка сохранения пропорций:")
print(f"Исходная доля кликов: {y.mean():.4f}")
print(f"Обучающая доля кликов: {y_train.mean():.4f}")
print(f"Тестовая доля кликов: {y_test.mean():.4f}")

**Разделение выполнено корректно:**
- Исходный датасет (50 000 строк) разделён на обучающую (40 000 строк, 80%) и тестовую (10 000 строк, 20%) выборки.
- Использована стратификация (`stratify=y`), что обеспечило сохранение пропорции классов во всех выборках.
- Доля кликов (CTR) во всех трёх выборках осталась одинаковой (≈0.172), что подтверждает корректность разделения.

Данные успешно разделены. Обучающая выборка готова для построения пайплайнов и обучения моделей. Тестовая выборка сохранена для финальной оценки качества.

## [Этап 4. Предобработка данных — построение пайплайнов <a id=4></a>](#content)

#### 4.1 Создайте пайплайн для предобработки данных

**Для числовых признаков:**
- Корректно заполните пропуски — средним, медианой или другим методом.
- Масштабируйте данные с помощью `StandardScaler`.
- Обработайте выбросы, если необходимо.

**Для категориальных признаков:**
- Корректно заполните пропуски — значением по умолчанию или модой.
- Примените кодирование:
  - One-Hot Encoding для признаков с малой кардинальностью.
  - Target Encoding для признаков с высокой кардинальностью.

#### 4.2 Объедините пайплайны
- Используйте `sklearn.pipeline.Pipeline` и `ColumnTransformer`.
- **Важно:** используйте информацию о пропусках и категориях только из обучающей выборки!

In [ ]:
# Создаём пайплайны для каждой группы
# Пайплайн для категориальных признаков
category_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('target', TargetEncoder(smoothing=30))
])

# Пайплайн для числовых признаков
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Пайплайн для low категориальных признаков
low_card_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

# Пайплайн для high категориальных признаков
# TargetEncoder заменяет категории на среднее значение целевой переменной
high_card_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('target', TargetEncoder(smoothing=30))
])

In [ ]:
# Объединяем всё в ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        #('cat', category_transformer, all_categorical)
        ('low_cat', low_card_transformer, low_categorical),
        ('high_cat', high_card_transformer, high_categorical)
    ],
    remainder='drop' 
)

## [Этап 5. Отбор признаков <a id=5></a>](#content)

#### 5.1 Примените фильтрационные методы
- Посчитайте корреляцию каждого признака с целевой переменной.
- Отберите топ лучших признаков. Объясните, почему остановились именно на таком количестве признаков.
- Удалите признаки с очень низкой вариацией `VarianceThreshold`.

#### 5.2 Примените методы-обёртки
- Используйте методы-обёртки для поиска оптимального набора признаков.

#### 5.3 Выберите финальный набор признаков
- Объедините результаты методов.
- Выберите признаки, которые прошли фильтрацию.

In [ ]:
# Производим предобработку обучающей выборки
X_train_preproc = preprocessor.fit_transform(X_train, y_train)

# Получаем имена столбцов после предобработки
feature_names = preprocessor.get_feature_names_out()

# Создаём DataFrame с признаками и целевой переменной
df_train_processed = pd.DataFrame(X_train_preproc, columns=feature_names)
df_train_processed['click'] = y_train.values

In [ ]:
# Рассчитываем корреляцию Пирсона
correlations = df_train_processed.corr()['click'].drop('click').sort_values(ascending=False)

# Отбираем признаки с малой корреляцией
corr_threshold = 0.02  # порог корреляции
selected_by_corr = correlations[abs(correlations) > corr_threshold].index.tolist()

print(f"\nОтобрано признаков с |корреляция| > {corr_threshold}: {len(selected_by_corr)} шт.")
print(*selected_by_corr, sep='\n')

In [ ]:
# Обучаем RFECV 

# с логистической регрессией
rfecv_selector = RFECV(
        estimator=LogisticRegression(solver='liblinear', class_weight='balanced', random_state=random_state),
        min_features_to_select=len(selected_by_corr),
        step=5,
        cv=StratifiedKFold(3))

# с SVC
rfecv_selector = RFECV(
        estimator=SVC(kernel='linear', random_state=random_state, max_iter=1000),
        min_features_to_select=len(selected_by_corr),
        step=5,
        cv=StratifiedKFold(3))

## [Этап 6. Обучение базовой модели <a id=6></a>](#content)

### 6.1 Обучите `DummyClassifier`
- Это нужно, чтобы обозначить самый простой базовый уровень работы модели.

### 6.2 Обучите `LogisticRegression`
- Используйте для обучения отобранные признаки.
- Примените кросс-валидацию на 5 фолдах.
- Посчитайте метрику PR-AUC. При необходимости дополнительно рассчитайте Precision, Recall и F1-score.
- Напоминаем, что для корректной кросс-валидации, предобработку нужно объединить с классификатором в Pipeline.

### 6.3 Обучите `SVC`

- Обучите SVC линейным ядром.
- Примените кросс-валидацию на 5 фолдах и посчитайте ту же метрику PR-ROC. При необходимости дополнительно рассчитайте Precision, Recall и F1-score.
- Калибровку модели мы проведём далее, поэтому здесь нужна модель `probability=False`

### 6.4 Сравните модели
- Убедитесь, что `LogisticRegression` работает лучше `DummyClassifier`.
- Сравните качество `LogisticRegression` с `SVC`.

In [ ]:
# Настройки кросс-валидации
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=random_state)

In [ ]:
scoring = {
    'precision': 'precision',           
    'recall': 'recall',
    'f1': 'f1',         
    'average_precision': 'average_precision'
}

In [ ]:
# Обучаем DummyClassifier
pipe_dummy = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', DummyClassifier(strategy='stratified', random_state=random_state))
])

# Кросс-валидация для DummyClassifier, используем accuracy для оценки
dummy_scores = cross_validate(pipe_dummy, X_train, y_train, 
                              cv=cv, 
                              scoring=scoring,
                              return_train_score=False)

print(f"DummyClassifier (кросс-валидация, 5 folds):")
print(f"  Precision:  {dummy_scores['test_precision'].mean():.4f} (+/- {dummy_scores['test_precision'].std():.4f})")
print(f"  Recall:     {dummy_scores['test_recall'].mean():.4f} (+/- {dummy_scores['test_recall'].std():.4f})")
print(f"  F1:     {dummy_scores['test_f1'].mean():.4f} (+/- {dummy_scores['test_f1'].std():.4f})")
print(f"  average_precision:   {dummy_scores['test_average_precision'].mean():.4f} (+/- {dummy_scores['test_average_precision'].std():.4f})")

In [ ]:
# Обучаем LogisticRegression
pipe_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=random_state))
])

lr_cv_results = cross_validate(pipe_lr, X_train, y_train, 
                               cv=cv, 
                               scoring=scoring,
                               return_train_score=False)

print(f"LogisticRegression результаты (5-fold CV):")
print(f"  Precision:  {lr_cv_results['test_precision'].mean():.4f} (+/- {lr_cv_results['test_precision'].std():.4f})")
print(f"  Recall:     {lr_cv_results['test_recall'].mean():.4f} (+/- {lr_cv_results['test_recall'].std():.4f})")
print(f"  F1:     {dummy_scores['test_f1'].mean():.4f} (+/- {dummy_scores['test_f1'].std():.4f})")
print(f"  average_precision:   {lr_cv_results['test_average_precision'].mean():.4f} (+/- {lr_cv_results['test_average_precision'].std():.4f})")

In [ ]:
# Обучаем SVC с линейным ядром
pipe_svc = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', SVC(kernel='linear', random_state=random_state, probability=True, max_iter=10000))
])


# Кросс-валидация для SVC
svc_cv_results = cross_validate(pipe_svc, X_train, y_train, 
                                cv=cv, 
                                scoring=scoring,
                                return_train_score=False)

print(f"SVC (kernel='linear') результаты (5-fold CV):")
print(f"  Precision:  {svc_cv_results['test_precision'].mean():.4f} (+/- {svc_cv_results['test_precision'].std():.4f})")
print(f"  Recall:     {svc_cv_results['test_recall'].mean():.4f} (+/- {svc_cv_results['test_recall'].std():.4f})")
print(f"  F1:     {dummy_scores['test_f1'].mean():.4f} (+/- {dummy_scores['test_f1'].std():.4f})")
print(f"  average_precision:   {svc_cv_results['test_average_precision'].mean():.4f} (+/- {svc_cv_results['test_average_precision'].std():.4f})")

In [ ]:
print("\nСравнение по метрике PR-AUC (5-fold CV):")
print(f"  DummyClassifier:     {dummy_scores['test_average_precision'].mean():.4f} (+/- {dummy_scores['test_average_precision'].std():.4f})")
print(f"  LogisticRegression:  {lr_cv_results['test_average_precision'].mean():.4f} (+/- {lr_cv_results['test_average_precision'].std():.4f})")
print(f"  SVC (linear):        {svc_cv_results['test_average_precision'].mean():.4f} (+/- {svc_cv_results['test_average_precision'].std():.4f})")

## [Этап 7. Подбор гиперпараметров: Grid Search с кросс-валидацией <a id=7></a>](#content)

#### 7.1 Определите сетку гиперпараметров
Определите ключевые параметры, которые влияют на качество моделей `LogisticRegression` и `SVC`.

#### 7.2 Примените Grid Search
- Используйте `GridSearchCV` для перебора всех комбинаций.
- Используйте `scoring='average_precision'`.
- Выведите лучшие параметры и их метрики.

#### 7.3 Составьте таблицу результатов
- Покажите топ-10 конфигураций с их метриками.

In [ ]:
# Параметры для перебора параметров LogisticRegression
param_grid_lr ={
    'model__C': [0.01, 0.1, 1, 10],
    'model__solver': ['liblinear', 'saga'],
    'model__penalty': ['l1', 'l2'],
    'model__max_iter': [100, 500, 1000]
}


In [ ]:
# Создаём модель LogisticRegression
pipe_lr_gs = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('feature_selector', rfecv_selector),
    ('model', LogisticRegression(solver='liblinear', max_iter=5000, class_weight='balanced', random_state=random_state))
])

In [ ]:
# Grid Search
grid_lr = GridSearchCV(
    estimator=pipe_lr_gs,
    param_grid=param_grid_lr,
    scoring='average_precision',
    cv=cv,
    n_jobs=-1,
    verbose=1
)

In [ ]:
# Выполняется поиск лучших параметров для LogisticRegression
grid_lr.fit(X_train, y_train)

In [ ]:
print("\nРезультаты Grid Search для LogisticRegression:")
print(f"Лучшие параметры: {grid_lr.best_params_}")
print(f"Лучший PR-AUC (average_precision): {grid_lr.best_score_:.4f}")

# Получаем и сортируем результаты
results_lr = grid_lr.cv_results_
scores_lr = results_lr['mean_test_score']
params_lr = results_lr['params']

# Создаем список пар (score, params) и сортируем по убыванию score
sorted_results_lr = sorted(zip(scores_lr, params_lr), key=lambda x: x[0], reverse=True)

# Выводим топ-10 комбинаций
print("\nТоп-10 комбинаций параметров (отсортированных по mean_test_score):")
for i in range(min(10, len(sorted_results_lr))):
    score, param = sorted_results_lr[i]
    print(f"  {i+1}. params={param}, PR-AUC={score:.4f}")

# Сохраняем лучшую модель LogisticRegression
best_lr = grid_lr.best_estimator_

In [ ]:
# Параметры для перебора параметров SVM
param_grid_svc = {
    'model__kernel': ['linear', 'rbf', 'poly',  'sigmoid', 'precomputed'],
    'model__C': [0.001, 0.01, 0.1, 1, 10],
    'model__gamma': ['scale', 'auto'],
    'model__degree': [2, 3, 4]
}

In [ ]:
# Создаём модель SVC
pipe_svc_gs = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('feature_selector', rfecv_selector),
    ('model', SVC(class_weight='balanced', random_state=random_state, max_iter=10000))
])

In [ ]:
# Grid Search
grid_svc = GridSearchCV(
    estimator=pipe_svc_gs,
    param_grid=param_grid_svc,
    scoring='average_precision',
    cv=cv,
    n_jobs=-1,
    verbose=1
)

In [ ]:
# Выполняется поиск лучших параметров для SVC
grid_svc.fit(X_train, y_train)

In [ ]:
print("\nРезультаты Grid Search для SVC:")
print(f"Лучшие параметры: {grid_svc.best_params_}")
print(f"Лучший PR-AUC (average_precision): {grid_svc.best_score_:.4f}")

# Получаем и сортируем результаты
results_svc = grid_svc.cv_results_
scores_svc = results_svc['mean_test_score']
params_svc = results_svc['params']

# Создаем список пар (score, params) и сортируем по убыванию score
sorted_results_svc = sorted(zip(scores_svc, params_svc), key=lambda x: x[0], reverse=True)

# Выводим топ-10 комбинаций
print("\nТоп-10 комбинаций параметров (отсортированных по mean_test_score):")
for i in range(min(10, len(sorted_results_svc))):
    score, param = sorted_results_svc[i]
    print(f"  {i+1}. params={param}, PR-AUC={score:.4f}")

# Сохраняем лучшую модель
best_svc = grid_svc.best_estimator_

In [ ]:
print("\nЛучшие результаты после Grid Search:")
print(f"  LogisticRegression: PR-AUC = {grid_lr.best_score_:.4f} (C={grid_lr.best_params_['model__C']})")
print(f"  SVC (linear):       PR-AUC = {grid_svc.best_score_:.4f} (C={grid_svc.best_params_['model__C']})")

if grid_lr.best_score_ > grid_svc.best_score_:
    print(f"\n✓ Лучшая модель: LogisticRegression с PR-AUC = {grid_lr.best_score_:.4f}")
    best_model = best_lr
    best_model_name = "LogisticRegression"
else:
    print(f"\n✓ Лучшая модель: SVC с PR-AUC = {grid_svc.best_score_:.4f}")
    best_model = best_svc
    best_model_name = "SVC"

## [Этап 8. Финальная модель <a id=8></a>](#content)

#### 8.1 Обучите финальную модель
- Используйте лучшие параметры из Grid Search.
- Обучите модели на всей обучающей выборке.

#### 8.2 Посчитайте метрики на тестовой выборке
- Необходимые метрики:
  - PR-AUC.
  - Оценка Бриера.
  - Дополнительные метрики при необходимости.

#### 8.3 Проанализируйте веса модели
- Выведите самые важные признаки по модулю коэффициентов.
- Интерпретируйте результаты.

In [ ]:
# Получаем предсказания
y_pred_proba = best_model.predict_proba(X_test)[:, 1]
y_pred = best_model.predict(X_test)

In [ ]:
# PR-AUC (average precision)
pr_auc = average_precision_score(y_test, y_pred_proba)

# Оценка Бриера (Brier Score)
brier = brier_score_loss(y_test, y_pred_proba)

# Дополнительные метрики
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

In [ ]:
print("Результаты на тестовой выборке:")
print("-" * 40)
print(f"  PR-AUC (average_precision):  {pr_auc:.4f}")
print(f"  Оценка Бриера (Brier Score): {brier:.4f}")
print(f"  Precision:                   {precision:.4f}")
print(f"  Recall:                      {recall:.4f}")

# Сравнение с результатами кросс-валидации
print("\nСравнение с кросс-валидацией:")
print(f"  PR-AUC (CV):     {grid_lr.best_score_:.4f}")
print(f"  PR-AUC (тест):   {pr_auc:.4f}")
print(f"  Разница:         {pr_auc - grid_lr.best_score_:.4f}")

if pr_auc >= grid_lr.best_score_:
    print("  ✓ Модель хорошо обобщается на тестовых данных")
else:
    print("  ⚠ Модель показывает чуть худший результат на тесте")

In [ ]:
# Получаем шаг RFECV
rfecv_step = best_model.named_steps['feature_selector']  # замените на реальное имя шага
# Или если RFECV встроен в классификатор, уточните структуру

# Отобранные признаки (имена исходных признаков, поданных на вход RFECV)
all_feature_names = best_model.named_steps['preprocessor'].get_feature_names_out()  # если есть препроцессинг
# Или напрямую, если знаете исходные колонки

# Маска отобранных признаков
support_mask = rfecv_step.support_  # булевый массив
selected_features = all_feature_names[support_mask]

# Коэффициенты модели (уже соответствуют отобранным признакам)
coefficients = best_model.named_steps['model'].coef_[0]

# Создаём DataFrame с признаками и их весами
feature_importance = pd.DataFrame({
    'feature': selected_features,
    'coefficient': coefficients,
    'abs_coefficient': np.abs(coefficients)
}).sort_values('abs_coefficient', ascending=False)

print("Топ-10 самых важных признаков (по модулю коэффициента):")
print("-" * 60)
for i, row in feature_importance.head(10).iterrows():
    print(f"  {row['feature']:30s} {row['coefficient']:10.4f}")

## [Этап 9. Калибровка модели <a id=9></a>](#content)

#### 9.1 Проверьте текущую калибровку
- Постройте калибровочную кривую, используйте `sklearn.calibration.calibration_curve`.
- Для обработки «сырых» значений SVC, нужно применить стандартную (необученную) сигмоиду для получения [0, 1].

#### 9.2 Примените методы калибровки
- Используйте `CalibratedClassifierCV` с методом `'isotonic'`.
- **Важно:** используйте для процедуры отдельную калибровочную выборку!

#### 9.3 Сравните модели до и после калибровки
- Посчитайте оценки Бриера для моделей до и после калибровки.
- Дополнительно можете рассчитать ECE и MCE для моделей до и после калибровки.
- Визуализируйте калибровочные кривые для моделей до и после калибровки.

## [Этап 10. Оценка качества калибровки <a id=10></a>](#content)

#### 10.1 Посчитайте метрики калибровки
- Оценка Бриера — средняя ошибка предсказанной вероятности.
- Дополнительная метрика ECE: среднее расхождение вероятностей.
- Дополнительная метрика MCE: максимальное расхождение вероятностей.

#### 10.2 Сравните модели до и после калибровки
- Выведите все метрики в одной таблице.
- Сделайте вывод о том, улучшила ли калибровка качество моделей.

## [Этап 11. Финальный отчёт и выводы <a id=11></a>](#content)

### 11.1 Сведите все результаты в таблицу

Покажите:
- Характеристики базовой модели `DummyClassifier`.
- Характеристики финальной модели.
- Метрики до и после калибровки.
- Топ-5 самых важных признаков.

### 11.2 Напишите выводы

Ответьте на вопросы:
- Улучшилось ли качество модели по сравнению с базовой?
- Какие признаки больше всего влияют на вероятность клика?
- Насколько хорошо модель откалибрована?
- Готова ли модель к использованию в продакшене?

### 11.3 Рекомендации

- Какие возможности улучшения модели вы видите?

## [Этап 12. Сохранение модели для продакшена <a id=12></a>](#content)

### 12.1 Сохраните артефакты

Сохраните:
1. пайплайн предобработки данных `preprocessor`;
2. финальную модель `calibrated_model`;
3. информацию о выбранных признаках.

### 12.2 Проверьте работоспособность вашего кода

- Загрузите сохранённые артефакты.
- Сделайте предсказания на новых данных.
- Убедитесь, что результаты совпадают.